# MCP Connectivity Tests

Validate connectivity and tool discovery for the remote MCP servers using the official MCP Python SDK.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = (
    Path.cwd().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd()
)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project Root:", PROJECT_ROOT)

Project Root: c:\Users\praya\Desktop\Tarvel_Agent\trip_planner


In [2]:
from config.settings import settings

SERVERS = [
    {"name": "Kiwi MCP", "url": settings.kiwi_mcp_server_url},
    {"name": "Agentorist MCP", "url": settings.agentorist_mcp_server_url},
]

for server in SERVERS:
    print(f"{server['name']}: {server['url']}")

ModuleNotFoundError: No module named 'pydantic_settings'

In [4]:
results = await run_tests()

for result in results:
    print("=" * 60)
    print("Server:", result["server"])
    print("URL:", result["url"])
    print("Connection:", result["status"])
    print("Available tools:", result["tools"])

Server: Kiwi MCP
URL: https://mcp.kiwi.com
Connection: connected
Available tools: ['search-flight', 'feedback-to-devs']
Server: Agentorist MCP
URL: https://mcp.agentorist.com/mcp
Connection: connected
Available tools: ['list_verticals', 'search', 'search_all', 'find_options', 'book', 'list_venues', 'request_unsupported_booking']


In [22]:
from pprint import pprint
from tools.hotel_tools import search_local_places

result = search_local_places("Mumbai", venue="DY Patil Stadium")

pprint(result)

Agentorist MCP payload for search: {'vertical': 'hotels', 'query': 'best hotels', 'location': 'Mumbai', 'agent_client': 'TripPlanner'}
{'available_tools': ['list_verticals',
                     'search',
                     'search_all',
                     'find_options',
                     'book',
                     'list_venues',
                     'request_unsupported_booking'],
 'data': {'content': [{'text': '{"vertical":"","results":[],"result_count":0,"bookable_count":0,"error":"internal_error","error_message":"An '
                               'internal error occurred. Please retry."}',
                       'type': 'text'}],
          'structured': {'bookable_count': 0,
                         'error': 'internal_error',
                         'error_message': 'An internal error occurred. Please '
                                          'retry.',
                         'result_count': 0,
                         'results': [],
                         'vertic

In [32]:
import asyncio

from mcp import ClientSession
from mcp.client.streamable_http import streamable_http_client

async def test_local_search():
    async with streamable_http_client("https://mcp.agentorist.com/mcp") as (
        read_stream,
        write_stream,
        _,
    ):
        async with ClientSession(read_stream, write_stream) as session:
            await session.initialize()

            result = await session.call_tool(
                "search",
                {
                    "vertical": "local",
                    "query": "restaurants",
                    "location": "Miami",
                    "agent_client": "TripPlanner"
                }
            )

            print(result)

await test_local_search()

meta=None content=[TextContent(type='text', text='{"vertical":"local","query":"restaurants","location":"Miami","results":[{"business_id":"UXHxLN3DcDGI57uDIfCuJA","name":"Old\'s Havana Cuban Bar & Cocina","categories":["cuban","bars","venues"],"rating":4.4,"review_count":3217,"price":"$$","address":"[redacted-pii]","phone":"[redacted-pii]","distance_m":7039,"is_closed":false,"image_url":"https://s3-media0.fl.yelpcdn.com/bphoto/OyMD-xvBjobfDmDdBs2Jfw/o.jpg","yelp_url":"https://www.yelp.com/biz/olds-havana-cuban-bar-and-cocina-miami?adjust_creative=DzZmti2q5-VcwEdSD2XcyQ&utm_campaign=yelp_api_v3&utm_medium=api_v3_business_search&utm_source=DzZmti2q5-VcwEdSD2XcyQ","booking_url":"https://www.yelp.com/biz/olds-havana-cuban-bar-and-cocina-miami?adjust_creative=DzZmti2q5-VcwEdSD2XcyQ&utm_campaign=yelp_api_v3&utm_medium=api_v3_business_search&utm_source=DzZmti2q5-VcwEdSD2XcyQ","supports_reservations":false,"supports_delivery":false,"supports_pickup":true,"bookable":false},{"business_id":"oxtMfB

In [40]:
import requests
from bs4 import BeautifulSoup

html = requests.get("https://gribstream.com/mcp").text

for keyword in [
    "Authorization",
    "Bearer",
    "API",
    "apikey",
    "api-key",
    "token",
    "streamable",
    "mcp"
]:
    if keyword.lower() in html.lower():
        print("FOUND:", keyword)


FOUND: API
FOUND: token
FOUND: streamable
FOUND: mcp


In [41]:
from bs4 import BeautifulSoup

html = requests.get("https://gribstream.com/mcp").text
soup = BeautifulSoup(html, "html.parser")

text = soup.get_text("\n")

for line in text.splitlines():
    line = line.strip()
    if any(
        word in line.lower()
        for word in [
            "api",
            "key",
            "token",
            "authorization",
            "bearer",
            "mcp"
        ]
    ):
        print(line)

GribStream MCP Connector
OpenAPI
GribStream MCP connector
This is the hosted GribStream MCP endpoint for AI tools that support remote MCP over Streamable HTTP.
https://gribstream.com/mcp
The endpoint uses GribStream OAuth. After you sign in, you approve the connector and choose the active API token it should use for live
queries through the regular GribStream API. The raw API token is not shown to the AI client.
ChatGPT: configure a custom MCP connector using Streamable HTTP.
Gemini CLI: add this URL as an MCP server in
MCP announcement blog post
OpenAPI spec


In [5]:
from config.settings import settings

print("Weather Provider:", getattr(settings, "weather_provider", "<not set>"))
print("Weather MCP:", getattr(settings, "weather_mcp_server_url", "<not set>"))
print("Weather MCP:", settings.weather_mcp_server_url)

Weather Provider: livedatalink
Weather MCP: https://livedatalink.ai/mcp
Weather MCP: https://livedatalink.ai/mcp


In [6]:
from mcp import ClientSession
from mcp.client.streamable_http import streamable_http_client
from config.settings import settings

async def test_kiwi_direct():

    payload = {
        "flyFrom": "Delhi",
        "flyTo": "Mumbai",
        "departureDate": "15/08/2026"
    }

    async with streamable_http_client(
        settings.kiwi_mcp_server_url
    ) as (
        read_stream,
        write_stream,
        _
    ):

        async with ClientSession(
            read_stream,
            write_stream
        ) as session:

            await session.initialize()

            result = await session.call_tool(
                "search-flight",
                payload
            )

            print(result)

await test_kiwi_direct()

meta=None content=[TextContent(type='text', text='[\n  {\n    "flyFrom": "DEL",\n    "flyTo": "BOM",\n    "cityFrom": "New Delhi",\n    "cityTo": "Mumbai",\n    "departure": {\n      "utc": "2026-08-14T22:30:00.000Z",\n      "local": "2026-08-15T04:00:00.000"\n    },\n    "arrival": {\n      "utc": "2026-08-15T00:45:00.000Z",\n      "local": "2026-08-15T06:15:00.000"\n    },\n    "totalDurationInSeconds": 8100,\n    "durationInSeconds": 8100,\n    "price": 64,\n    "deepLink": "https://on.kiwi.com/6Jx7Qu",\n    "currency": "EUR"\n  },\n  {\n    "flyFrom": "DEL",\n    "flyTo": "BOM",\n    "cityFrom": "New Delhi",\n    "cityTo": "Mumbai",\n    "departure": {\n      "utc": "2026-08-15T03:00:00.000Z",\n      "local": "2026-08-15T08:30:00.000"\n    },\n    "arrival": {\n      "utc": "2026-08-15T05:15:00.000Z",\n      "local": "2026-08-15T10:45:00.000"\n    },\n    "totalDurationInSeconds": 8100,\n    "durationInSeconds": 8100,\n    "price": 70,\n    "deepLink": "https://on.kiwi.com/sMa0fJ",

In [5]:
from tools.flight_tools import search_flights
from pprint import pprint

result = search_flights(
    origin="Delhi",
    destination="Mumbai",
    event_date="2026-08-15",
)

print("\nRESULT TYPE:")
print(type(result))

print("\nRESULT:")
pprint(result)

TOOL SCHEMA:
{'type': 'object', 'properties': {'flyFrom': {'type': 'string', 'minLength': 1, 'description': 'Location to fly from: It could be a city or an airport name or code'}, 'flyTo': {'type': 'string', 'description': 'Location to fly to: It could be a city or an airport name or code'}, 'departureDate': {'type': 'string', 'pattern': '^\\d{2}\\/\\d{2}\\/\\d{4}$', 'description': 'Departure date in dd/mm/yyyy format'}, 'departureDateFlexRange': {'type': 'integer', 'minimum': 0, 'maximum': 3, 'default': 0, 'description': 'Departure date flexibility range in days (0 to 3 days before/after the selected departure date)'}, 'returnDate': {'type': 'string', 'pattern': '^\\d{2}\\/\\d{2}\\/\\d{4}$', 'description': 'Return date in dd/mm/yyyy format'}, 'returnDateFlexRange': {'type': 'integer', 'minimum': 0, 'maximum': 3, 'default': 0, 'description': 'Return date flexibility range in days (0 to 3 days before/after the selected return date)'}, 'passengers': {'type': 'object', 'properties': {'adu

In [9]:
from graph.trip_graph import build_trip_graph

print("Import Successful")

Import Successful


In [10]:
from graph.trip_graph import build_trip_graph

graph = build_trip_graph()

print(graph)

In [11]:
from graph.trip_graph import build_trip_graph

graph = build_trip_graph()

print(type(graph))

<class 'langgraph.graph.state.CompiledStateGraph'>


In [3]:
from agents.itinerary_agent import itinerary_agent

state = {
    "destination": "New York",
    "venue": "Madison Square Garden",
    "event_date": "2026-08-15",
    "supervisor_notes": "Trip planning in progress.",
    "flight_notes": "Flight found from Miami to New York.",
    "hotel_notes": "Hotel found near Madison Square Garden.",
    "weather_notes": "Weather data retrieved successfully.",
    "search_notes": "Popular attractions identified.",
    "errors": [],
}

result = itinerary_agent(state)

print("Itinerary Status:", result.get("itinerary_status"))
print("Errors:", result.get("errors"))
print("Itinerary Present:", bool(result.get("itinerary")))

Itinerary Status: completed
Errors: []
Itinerary Present: True


In [9]:
from state.trip_state import TripPlannerState

print("flight_status" in TripPlannerState.__annotations__)
print("hotel_status" in TripPlannerState.__annotations__)
print("weather_status" in TripPlannerState.__annotations__)
print("search_status" in TripPlannerState.__annotations__)
print("itinerary_status" in TripPlannerState.__annotations__)

True
True
True
True
True


In [3]:
from langgraph.checkpoint.sqlite import SqliteSaver

print("SQLite checkpointing available")

SQLite checkpointing available


In [5]:
from langgraph.checkpoint.sqlite import SqliteSaver

print(SqliteSaver)
print(hasattr(SqliteSaver, "from_conn_string"))

<class 'langgraph.checkpoint.sqlite.SqliteSaver'>
True


In [5]:
from langgraph.checkpoint.sqlite import SqliteSaver

try:
    saver = SqliteSaver.from_conn_string("trip_planner.db")
    print("SUCCESS")
    print(type(saver))
except Exception as e:
    print("FAILED")
    print(type(e).__name__, e)

SUCCESS
<class 'contextlib._GeneratorContextManager'>


In [8]:
from langgraph.checkpoint.sqlite import SqliteSaver
from pathlib import Path

db_path = Path("trip_planner.db").resolve()

print("DB PATH:")
print(db_path)
print()

cm = SqliteSaver.from_conn_string(str(db_path))

print("CONTEXT MANAGER CREATED")

try:
    saver = cm.__enter__()
    print("ENTER SUCCESS")
    print(type(saver))
except Exception as e:
    print("ENTER FAILED")
    print(type(e).__name__)
    print(e)

DB PATH:
C:\Users\praya\Desktop\Tarvel_Agent\trip_planner\notebooks\trip_planner.db

CONTEXT MANAGER CREATED
ENTER SUCCESS
<class 'langgraph.checkpoint.sqlite.SqliteSaver'>


In [10]:
from memory.sqlite_checkpoint import get_checkpointer

cp = get_checkpointer()

print(type(cp))

<class 'langgraph.checkpoint.sqlite.SqliteSaver'>


In [11]:
from graph.trip_graph import build_trip_graph

graph = build_trip_graph()

print("GRAPH COMPILED")

GRAPH COMPILED


In [7]:
from graph.trip_graph import build_trip_graph

graph = build_trip_graph()

snapshot = graph.get_state(
    {
        "configurable": {
            "thread_id": "trip_test_001"
        }
    }
)

print(snapshot.values["destination"])
print(snapshot.values["status"])

New York
completed


In [4]:
state = result.get("state", {})

print("=" * 60)
print("PHASE 25 FINAL CHECK")
print("=" * 60)

print("conversation_status =", result.get("status"))

print("flight_status       =", state.get("flight_status"))
print("hotel_status        =", state.get("hotel_status"))
print("weather_status      =", state.get("weather_status"))
print("itinerary_status    =", state.get("itinerary_status"))

print("origin              =", state.get("origin"))
print("destination         =", state.get("destination"))
print("event_date          =", state.get("event_date"))

PHASE 25 FINAL CHECK
conversation_status = completed
flight_status       = completed
hotel_status        = completed
weather_status      = completed
itinerary_status    = completed
origin              = Miami
destination         = New York
event_date          = 2026-08-15


In [5]:
print(result.get("itinerary_status"))

completed


In [7]:
from agents.report_formatter_agent import report_formatter_agent

test_state = {
    "origin": "Miami",
    "destination": "New York",
    "event_date": "2026-08-15",
    "venue": "Madison Square Garden",
    "flight_notes": "Test flight",
    "hotel_notes": "Test hotel",
    "weather_notes": "Test weather",
    "search_notes": "Test search",
    "itinerary": "Test itinerary",
}

report = report_formatter_agent(test_state)

print(type(report))
print(report)

<class 'str'>
Here's where things stand right now

Trip Summary
- Route: Miami → New York
- Event Venue: Madison Square Garden
- Event Date: 2026-08-15

Flights
Test flight

Hotels
Test hotel

Weather
Test weather

Local Highlights
Test search

Suggested Itinerary
Test itinerary

Next Steps
- Review flight recommendations
- Confirm hotel selection
- Check weather before departure
- Finalize event plans
- Book all reservations

Travel Status
- Planning completed


In [4]:
from state.trip_state import TripPlannerState

print(sorted(TripPlannerState.__annotations__.keys()))

['destination', 'errors', 'event_date', 'final_report', 'flight_booking_link', 'flight_details', 'flight_notes', 'flight_status', 'hotel_booking_links', 'hotel_details', 'hotel_notes', 'hotel_price_details', 'hotel_status', 'itinerary', 'itinerary_status', 'origin', 'recommended_flight_price', 'recommended_hotel_price', 'search_notes', 'search_results', 'search_status', 'status', 'supervisor_notes', 'travelers', 'venue', 'weather_details', 'weather_notes', 'weather_status']


In [3]:
from agents.report_formatter_agent import report_formatter_agent

test_state = {
    "origin": "Miami",
    "destination": "New York",
    "venue": "Madison Square Garden",
    "event_date": "2026-08-15",
}

print(report_formatter_agent(test_state))

{'final_report': "# Here's where things stand right now\n\n## Trip Summary\n* Origin: Miami\n* Destination: New York\n* Event Venue: Madison Square Garden\n* Event Date: 2026-08-15\n\n## Flights\n* Route: Miami → New York\n* Departure: Information unavailable\n* Arrival: Information unavailable\n* Price: Information unavailable\n* Booking link: Information unavailable\n\n## Hotels\n* Hotel name: Information unavailable\n* Rating: Information unavailable\n* Nightly price: Information unavailable\n* Booking link: Information unavailable\n\n## Weather\n* Weather forecast: Information unavailable\n\n## Local Highlights\n* Information unavailable\n\n## Suggested Itinerary\n* Information unavailable\n\n## Next Steps\n* Research and book flights from Miami to New York\n* Find and book a hotel in New York\n* Check the weather forecast for New York\n* Explore local highlights and create a suggested itinerary"}


In [4]:
from tools.weather_tools import get_weather

result = get_weather(
    destination="New York",
    event_date="2026-06-10"
)

print(result)

{'status': 'success', 'provider': 'livedatalink', 'forecast': {'status': 'success', 'provider': 'livedatalink', 'tool_used': 'weather_forecast', 'data': '8-DAY FORECAST - New York, New York, United States\n════════════════════════════════════════════════════════\n\nDate        Hi    Lo    Conditions            Rain%  Wind\n────────────────────────────────────────────────────────\nToday       77°   49°   Overcast              2%     13 mph\nTomorrow    81°   53°   Overcast              2%     10 mph\nThu, Jun 4  81°   62°   Overcast              0%     15 mph\nFri, Jun 5  86°   67°   Overcast              1%     13 mph\nSat, Jun 6  89°   69°   Overcast              12%    14 mph\nSun, Jun 7  84°   71°   Moderate drizzle      38%    15 mph\nMon, Jun 8  70°   61°   Moderate drizzle      38%    18 mph\nTue, Jun 9  67°   61°   Slight rain showers   26%    15 mph\n\nSunrise: 09:26  |  Sunset: 00:21'}, 'air_quality': {'status': 'success', 'provider': 'livedatalink', 'tool_used': 'air_quality'

In [4]:
from tools.flight_tools import search_flights

result = search_flights(
    origin="London",
    destination="New York",
    event_date="2026-06-05",
    travelers=1,
)

print(result)

{'status': 'success', 'provider': 'kiwi', 'tool_used': 'search-flight', 'data': {'content': [{'type': 'text', 'text': '[\n  {\n    "flyFrom": "LTN",\n    "flyTo": "JFK",\n    "cityFrom": "London",\n    "cityTo": "New York",\n    "departure": {\n      "utc": "2026-06-05T04:40:00.000Z",\n      "local": "2026-06-05T05:40:00.000"\n    },\n    "arrival": {\n      "utc": "2026-06-06T02:30:00.000Z",\n      "local": "2026-06-05T22:30:00.000"\n    },\n    "totalDurationInSeconds": 78600,\n    "durationInSeconds": 78600,\n    "price": 387,\n    "deepLink": "https://on.kiwi.com/nhg6d2",\n    "currency": "EUR",\n    "layovers": [\n      {\n        "at": "BCN",\n        "city": "Barcelona",\n        "cityCode": "BCN",\n        "arrival": {\n          "utc": "2026-06-05T06:50:00.000Z",\n          "local": "2026-06-05T08:50:00.000"\n        },\n        "departure": {\n          "utc": "2026-06-05T08:55:00.000Z",\n          "local": "2026-06-05T10:55:00.000"\n        }\n      },\n      {\n        "at"

In [3]:
from services.trip_planner_service import plan_trip

trip_result = plan_trip(
    """
    Travel from Delhi to New York
    for a concert at Madison Square Garden
    on June 4, 2026.
    """
)

print(trip_result["final_report"])

from IPython.display import Markdown, display

display(Markdown(trip_result["final_report"]))

RUNNING NODE: supervisor_agent
RUNNING NODE: flight_agent
RUNNING NODE: hotel_agent
RUNNING NODE: weather_agent
RUNNING NODE: search_agent
RUNNING NODE: itinerary_agent
# Here's where things stand right now

## Trip Summary
- **Route:** Delhi → New York
- **Venue:** Madison Square Garden
- **Event Date:** 2026-06-04

## Flights
### Recommended Flight
#### Route:
New Delhi/DEL → New York/JFK

#### Departure:
2026-06-04T21:45:00.000

#### Arrival:
2026-06-05T09:40:00.000

#### Price:
706 EUR

#### Booking Link:
https://on.kiwi.com/GfL7nz

**Book here:** https://on.kiwi.com/GfL7nz

## Hotels
Based on the provided MCP Hotel Results, I recommend the following hotels for your stay in New York, considering the venue proximity to Madison Square Garden and the traveler's preferences:

1. **Casablanca Hotel**: 
   - Rating: 4.6
   - Price Category: $$
   - Address: 147 W 43rd St, New York, NY 10036
   - Book: https://www.yelp.com/biz/casablanca-hotel-new-york-2?adjust_creative=DzZmti2q5-VcwEdSD2

# Here's where things stand right now

## Trip Summary
- **Route:** Delhi → New York
- **Venue:** Madison Square Garden
- **Event Date:** 2026-06-04

## Flights
### Recommended Flight
#### Route:
New Delhi/DEL → New York/JFK

#### Departure:
2026-06-04T21:45:00.000

#### Arrival:
2026-06-05T09:40:00.000

#### Price:
706 EUR

#### Booking Link:
https://on.kiwi.com/GfL7nz

**Book here:** https://on.kiwi.com/GfL7nz

## Hotels
Based on the provided MCP Hotel Results, I recommend the following hotels for your stay in New York, considering the venue proximity to Madison Square Garden and the traveler's preferences:

1. **Casablanca Hotel**: 
   - Rating: 4.6
   - Price Category: $$
   - Address: 147 W 43rd St, New York, NY 10036
   - Book: https://www.yelp.com/biz/casablanca-hotel-new-york-2?adjust_creative=DzZmti2q5-VcwEdSD2XcyQ&utm_campaign=yelp_api_v3&utm_medium=api_v3_business_search&utm_source=DzZmti2q5-VcwEdSD2XcyQ
   This hotel is highly rated and has a moderate price category, making it a great option.

2. **Renaissance New York Midtown Hotel**: 
   - Rating: 4.0
   - Price Category: $$
   - Address: 218 West 35th Street, New York, NY 10001
   - Book: https://www.yelp.com/biz/renaissance-new-york-midtown-hotel-new-york?adjust_creative=DzZmti2q5-VcwEdSD2XcyQ&utm_campaign=yelp_api_v3&utm_medium=api_v3_business_search&utm_source=DzZmti2q5-VcwEdSD2XcyQ
   This hotel is close to Madison Square Garden and has a moderate price category.

3. **Motto by Hilton New York City Chelsea**: 
   - Rating: 4.0
   - Price Category: $$
   - Address: 113 W 24th St, New York, NY 10001
   - Book: https://www.yelp.com/biz/motto-by-hilton-new-york-city-chelsea-new-york?adjust_creative=DzZmti2q5-VcwEdSD2XcyQ&utm_campaign=yelp_api_v3&utm_medium=api_v3_business_search&utm_source=DzZmti2q5-VcwEdSD2XcyQ
   This hotel is also close to the venue and has a moderate price category.

These hotels offer a great balance of reputation, location, and price category. However, please note that the price category is subject to change, and it's always best to check the current prices before booking.

**Booking Links:**
- https://www.yelp.com/biz/casablanca-hotel-new-york-2?adjust_creative=DzZmti2q5-VcwEdSD2XcyQ&utm_campaign=yelp_api_v3&utm_medium=api_v3_business_search&utm_source=DzZmti2q5-VcwEdSD2XcyQ
- https://www.yelp.com/biz/park-hyatt-new-york-new-york?adjust_creative=DzZmti2q5-VcwEdSD2XcyQ&utm_campaign=yelp_api_v3&utm_medium=api_v3_business_search&utm_source=DzZmti2q5-VcwEdSD2XcyQ
- https://www.yelp.com/biz/hotel-50-bowery-new-york-2?adjust_creative=DzZmti2q5-VcwEdSD2XcyQ&utm_campaign=yelp_api_v3&utm_medium=api_v3_business_search&utm_source=DzZmti2q5-VcwEdSD2XcyQ
- https://www.yelp.com/biz/citizenm-new-york-times-square-hotel-new-york?adjust_creative=DzZmti2q5-VcwEdSD2XcyQ&utm_campaign=yelp_api_v3&utm_medium=api_v3_business_search&utm_source=DzZmti2q5-VcwEdSD2XcyQ
- https://www.yelp.com/biz/the-mercer-hotel-new-york?adjust_creative=DzZmti2q5-VcwEdSD2XcyQ&utm_campaign=yelp_api_v3&utm_medium=api_v3_business_search&utm_source=DzZmti2q5-VcwEdSD2XcyQ
- https://www.yelp.com/biz/the-wallace-new-york-3?adjust_creative=DzZmti2q5-VcwEdSD2XcyQ&utm_campaign=yelp_api_v3&utm_medium=api_v3_business_search&utm_source=DzZmti2q5-VcwEdSD2XcyQ
- https://www.yelp.com/biz/1-hotel-central-park-new-york?adjust_creative=DzZmti2q5-VcwEdSD2XcyQ&utm_campaign=yelp_api_v3&utm_medium=api_v3_business_search&utm_source=DzZmti2q5-VcwEdSD2XcyQ
- https://www.yelp.com/biz/arlo-nomad-new-york?adjust_creative=DzZmti2q5-VcwEdSD2XcyQ&utm_campaign=yelp_api_v3&utm_medium=api_v3_business_search&utm_source=DzZmti2q5-VcwEdSD2XcyQ
- https://www.yelp.com/biz/the-fifth-avenue-hotel-new-york-2?adjust_creative=DzZmti2q5-VcwEdSD2XcyQ&utm_campaign=yelp_api_v3&utm_medium=api_v3_business_search&utm_source=DzZmti2q5-VcwEdSD2XcyQ
- https://www.yelp.com/biz/the-hotel-chelsea-new-york?adjust_creative=DzZmti2q5-VcwEdSD2XcyQ&utm_campaign=yelp_api_v3&utm_medium=api_v3_business_search&utm_source=DzZmti2q5-VcwEdSD2XcyQ
- https://www.yelp.com/biz/1-hotel-brooklyn-bridge-brooklyn?adjust_creative=DzZmti2q5-VcwEdSD2XcyQ&utm_campaign=yelp_api_v3&utm_medium=api_v3_business_search&utm_source=DzZmti2q5-VcwEdSD2XcyQ
- https://www.yelp.com/biz/renaissance-new-york-midtown-hotel-new-york?adjust_creative=DzZmti2q5-VcwEdSD2XcyQ&utm_campaign=yelp_api_v3&utm_medium=api_v3_business_search&utm_source=DzZmti2q5-VcwEdSD2XcyQ
- https://www.yelp.com/biz/the-st-regis-new-york-new-york?adjust_creative=DzZmti2q5-VcwEdSD2XcyQ&utm_campaign=yelp_api_v3&utm_medium=api_v3_business_search&utm_source=DzZmti2q5-VcwEdSD2XcyQ
- https://www.yelp.com/biz/hampton-inn-brooklyn-downtown-brooklyn-2?adjust_creative=DzZmti2q5-VcwEdSD2XcyQ&utm_campaign=yelp_api_v3&utm_medium=api_v3_business_search&utm_source=DzZmti2q5-VcwEdSD2XcyQ
- https://www.yelp.com/biz/kimpton-hotel-eventi-new-york?adjust_creative=DzZmti2q5-VcwEdSD2XcyQ&utm_campaign=yelp_api_v3&utm_medium=api_v3_business_search&utm_source=DzZmti2q5-VcwEdSD2XcyQ
- https://www.yelp.com/biz/sanctuary-hotel-new-york-new-york?adjust_creative=DzZmti2q5-VcwEdSD2XcyQ&utm_campaign=yelp_api_v3&utm_medium=api_v3_business_search&utm_source=DzZmti2q5-VcwEdSD2XcyQ
- https://www.yelp.com/biz/motto-by-hilton-new-york-city-chelsea-new-york?adjust_creative=DzZmti2q5-VcwEdSD2XcyQ&utm_campaign=yelp_api_v3&utm_medium=api_v3_business_search&utm_source=DzZmti2q5-VcwEdSD2XcyQ
- https://www.yelp.com/biz/the-ludlow-new-york-city-new-york?adjust_creative=DzZmti2q5-VcwEdSD2XcyQ&utm_campaign=yelp_api_v3&utm_medium=api_v3_business_search&utm_source=DzZmti2q5-VcwEdSD2XcyQ
- https://www.yelp.com/biz/the-surrey-a-corinthia-hotel-new-york-2?adjust_creative=DzZmti2q5-VcwEdSD2XcyQ&utm_campaign=yelp_api_v3&utm_medium=api_v3_business_search&utm_source=DzZmti2q5-VcwEdSD2XcyQ
- https://www.yelp.com/biz/library-hotel-by-library-hotel-collection-new-york?adjust_creative=DzZmti2q5-VcwEdSD2XcyQ&utm_campaign=yelp_api_v3&utm_medium=api_v3_business_search&utm_source=DzZmti2q5-VcwEdSD2XcyQ

## Weather
**Weather Summary for New York on 2026-06-04:**

Since the event date (2026-06-04) is beyond the 2-day forecast range provided, I will summarize the current forecast as representative weather and provide historical climate expectations for New York in June.

**Current 2-Day Forecast:**
- Weather Conditions: Overcast
- Temperature Range: 49°F - 81°F (9°C - 27°C)
- Rain Chance: 1% - 2%

**Historical Climate Expectations for New York in June:**
June is typically a warm month in New York, with average highs around 77°F (25°C) and average lows around 63°F (17°C). The chance of rain is moderate, with an average of 12 rainy days throughout the month. 

**Air Quality:**
The current air quality in New York is Moderate, with an AQI (US) of 59. The pollutant levels are:
- PM2.5: 11.1 µg/m³
- PM10: 13.9 µg/m³
- Ozone: 99.0 µg/m³
- NO₂: 5.4 µg/m³
- SO₂: 1.2 µg/m³
- CO: 141.0 µg/m³

Please note that air quality can change rapidly and may vary by location within the city. It's always a good idea to check current conditions before heading out.

## Local Highlights
## Top Attractions
Madison Square Garden, Mustang Harry's, Manhattan West

## Recommended Restaurants
Carmine's Italian Restaurant, The Cloisters, The Levee, The Tippler, Tabata Noodle Restaurant, The Smith, New York Pizza Suprema, Magnolia Bakery, The Park

## Local Transportation
Subway, train, walking, NY Waterway ferry service

## Suggested Itinerary
**Complete Travel Itinerary**

**Destination:** New York
**Venue:** Madison Square Garden
**Event Date:** 2026-06-04

**Arrival Recommendations:**
- Book your flight from New Delhi (DEL) to New York (JFK) on June 4, 2026, at 21:45, arriving on June 5, 2026, at 09:40.
- Flight price: 706 EUR
- Booking link: https://on.kiwi.com/GfL7nz

**Hotel Check-in Suggestions:**
- Recommended hotels:
  1. **Casablanca Hotel**: 147 W 43rd St, New York, NY 10036, Rating: 4.6, Price Category: $$
  2. **Renaissance New York Midtown Hotel**: 218 West 35th Street, New York, NY 10001, Rating: 4.0, Price Category: $$
  3. **Motto by Hilton New York City Chelsea**: 113 W 24th St, New York, NY 10001, Rating: 4.0, Price Category: $$
- Book your preferred hotel through the provided Yelp links.

**Event-Day Guidance:**
- Attend the event at Madison Square Garden on June 4, 2026.
- Plan to arrive at the venue early to account for security checks and crowds.

**Local Highlights:**
- Top attractions: Madison Square Garden, Mustang Harry's, Manhattan West
- Recommended restaurants:
  1. Carmine's Italian Restaurant
  2. The Cloisters
  3. The Levee
  4. The Tippler
  5. Tabata Noodle Restaurant
  6. The Smith
  7. New York Pizza Suprema
  8. Magnolia Bakery
  9. The Park
- Local transportation options: Subway, train, walking, NY Waterway ferry service

**Dining Recommendations:**
- Try the recommended restaurants for a variety of cuisines and experiences.

**Weather and Air Quality:**
- Expect warm weather in June, with average highs around 77°F (25°C) and average lows around 63°F (17°C).
- Check current weather forecasts and air quality indexes before heading out.

**Return-Trip Planning:**
- Book your return flight according to your preferred departure date and time.
- Ensure you have all necessary travel documents and check the airline's baggage policy.

By following this itinerary, you'll be well-prepared for your trip to New York and the event at Madison Square Garden. Enjoy your travels!

## Next Steps
- Confirm and book your flight
- Confirm and book your hotel
- Review the weather forecast closer to the event date
- Follow the suggested itinerary on arrival
